In [ ]:
import pandas as pd
import matplotlib.pyplot as plt 
import numpy as np
import sys
sys.path.append('../../scripts')

pd.set_option('display.max_columns', None)

In [ ]:
csv_file = '../../data/raw/ape_marche_dataset.csv'

df = pd.read_csv(csv_file)
print(df.shape) # Rows, Columns

In [ ]:
df.head()

## Missing values

In [ ]:
## features with missing values
features_na = [feature for feature in df.columns if df[feature].isnull().sum() > 1]

## feature name + percentage of missing values
for feature in features_na:
    print(feature, np.round(df[feature].isnull().mean(), 5), '% missing values')

### Relationship between missing values and the traget variable (epglnren)

In [ ]:
for feature in features_na:
    data = df.copy()

    # 1 indicates missing value, 0 indicates not missing value
    data[feature] = np.where(data[feature].isnull(), 1, 0)
    data.groupby(feature)['epglnren'].median().plot.bar()
    plt.title(feature)
    plt.show()

## Numerical values

In [ ]:
numerical_features = df.select_dtypes(include=['number']).columns.tolist()
print('Numerical features:', len(numerical_features))

df[numerical_features].head()

## Temporal variables

In [ ]:
# only one time feature exists, and it is of no use for our analysis or prediction, so it will be dropped

time_features = ['scadenza', 'anno_installazione_risc']

# anno_costruzione is a numerical predictor, not a time feature, and is relevant for energy efficiency calculation, so it will not be dropped

### Relationship between anno_costruzione and epglnren

In [ ]:
df.groupby('anno_costruzione')['epglnren'].median().plot.bar()
plt.xlabel('anno_costruzione')
plt.ylabel('Median epglnren')
plt.title('Median epglnren by anno_costruzione')
plt.xticks(rotation=70, ha='right', fontsize=5)
plt.tight_layout()
plt.show()

The graph shows that there is no correlation between the year the building was built and non-renewable energy consumption

## Continuous and discrete variables

In [ ]:
discrete_features = [feature for feature in numerical_features if len(df[feature].unique()) < 15 and feature != 'ricambi_aria_ora']
print('Discrete features count: ', len(discrete_features))
discrete_features

In [ ]:
## exploring the relationship between discrete features and the target variable
for feature in discrete_features:
    data = df.copy()
    data.groupby(feature)['epglnren'].median().plot.bar()
    plt.xlabel(feature)
    plt.ylabel('epglnren')
    plt.title(feature)
    plt.show()

In [ ]:
## unwanted features
unwanted_features = time_features + ['xml_filename', 'file_timestamp']

In [ ]:
## text features
text_features = ['intervento_consigliato', 'tipo_riscaldamento']

In [ ]:
## categorical features
categorical_features = [feature for feature in df if feature in df.select_dtypes(include=['object','string']).columns.tolist() and feature not in unwanted_features and feature not in text_features]
categorical_features

In [ ]:
## continuous features
continuous_features = [feature for feature in numerical_features if feature not in discrete_features and feature not in unwanted_features and feature not in time_features]
print(f'continuous features count: {len(continuous_features)}')

In [ ]:
## histograms for continuous features
for feature in continuous_features:
    data = df.copy()
    data[feature].hist(bins=30)
    plt.xlabel(feature)
    plt.ylabel('Count')
    plt.title(feature)
    plt.show()

In [ ]:
## logarithmic transformation for skewed features
for feature in continuous_features:
    data = df.copy()
    data = data[(data[feature] > 0) & (data['epglnren'] > 0)]
    
    if len(data) > 0:
        data[feature] = np.log(data[feature])
        data['epglnren'] = np.log(data['epglnren'])
        plt.scatter(data[feature], data['epglnren'])
        plt.xlabel(feature)
        plt.ylabel('epglnren')
        plt.title(feature)
        plt.show()

## Outliers

In [ ]:
for feature in continuous_features:
    data = df.copy()
    data = data[(data[feature] > 0) & (data['epglnren'] > 0)]

    if len(data) > 0:
        data[feature] = np.log(data[feature])
        data.boxplot(column=feature)
        plt.ylabel(feature)
        plt.title(feature)
        plt.show()
    else:
        print(f'{feature} has no positive values, skipping.')

In [ ]:
## Categorical features
categorical_features = ['classe_energetica', 'classe_nuovi', 'classe_esistenti', 'classe_raggiungibile', 'zona_climatica']
categorical_features

In [ ]:
df[categorical_features].head()

In [ ]:
for feature in categorical_features:
    print(f'feature: {feature}, number of categories: {len(df[feature].unique())}')

In [ ]:
## relationship between categorical features and the target variable epglnren
for feature in categorical_features:
    df.groupby(feature)['epglnren'].median().plot.bar()
    plt.xlabel(feature)
    plt.ylabel('epglnren')
    plt.title(feature)
    plt.show()